In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
apple = yf.download(
    "AAPL",
    start="2020-01-01",
    end="2025-01-01"

)

In [ ]:
apple.columns = apple.columns.get_level_values(0)

In [ ]:
print(apple.columns)
print(type(apple))
print(type(apple["Close"]))

In [ ]:
def add_moving_averages(df):
  """
  Add 30-day and 50-day moving averages to a stock price DataFrame.
  """

  df["MA30"] = df["Close"].rolling(window=30).mean()
  df["MA50"] = df["Close"].rolling(window=50).mean()
  return df

In [ ]:
apple = add_moving_averages(apple)

In [ ]:
apple[["Close", "MA30", "MA50"]].tail()


# Generate trading positions

The trading cycle rule is based on the relationship between the 30-day and 50-day moving averages.

- Position = 1 when MA30 is above MA50
- Position = 0 when MA30 is below MA50

The position represents where the strategy is invested in the stock or holding cash.

In [ ]:
def generate_signals(df):
  """
  Generate trading positions from the moving averages.

  Position:
    1 = invested in the stock
    0 = holding cash
  """
  df["Position"] = (
      df["MA30"] > df["MA50"]
  ).astype(int)

  return df


In [ ]:
apple = generate_signals(apple)

In [ ]:
apple[["MA30", "MA50", "Position"]].tail(10)

In [ ]:
apple["Position"].value_counts()

# Step 3: Calculate Returns

We calculate the daily percentage returns of the stock. This represents the return an investor would receive from holding the stock through each trading day.

In [ ]:
def calculate_returns(df):
  """
  Calculate the daily stock returns and strategy returns

  Returns:
    Returns - daily stock return
    Strategy_returns - daily return earned by the strategy
  """
  df["Returns"] = df["Close"].pct_change()
  df["Strategy_Returns"] = (
      df["Returns"]
      * df["Position"].shift(1)
  )
  return df


In [ ]:
apple = calculate_returns(apple)

In [ ]:
apple[["Position", "Returns", "Strategy_Returns"]].head(10)

In [ ]:
apple[apple["Position"] == 1].head()

#Step 4: Simulate Portfolio Growth

We simulate the growth of an initial investment using the daily returns generated by the trading strategy.

The portfolio compounds each day's strategy return over the full backtest period.

In [ ]:
def calculate_portfolio(df, initial_capital= 10000):
  """
  Simulate portfolio growth form strategy returns.

  Parameters
  ---------
  df  : pandas.DataFrame
      DataFrame containing Strategy_Returns.
    initial_capital : float
      Starting portfolio value.

  Returns
  ---------
  pandas.DataFrame
      DataFrame with Portfolio value.

  """
  df["Portfolio"] = (
      initial_capital
      * ( 1 + df["Strategy_Returns"].fillna(0)).cumprod()
  )

  return df

In [ ]:
apple = calculate_portfolio(apple)

In [ ]:
apple[["Strategy_Returns", "Portfolio"]].tail()

In [ ]:
apple.loc[
    apple["Strategy_Returns"] != 0,
    ["Position", "Returns", "Strategy_Returns", "Portfolio"]
].head(10)

# Step 5: Calculate Buy-and-Hold Perfomance

We create a buy-and-hold benchmark to evalaute the moving average strategy agaisnt a passive investment.

The benchmark remains fully invested thoughout the backtest, so its portfolio value reflects the cumulative performance of the stock itself.

In [ ]:
def calculate_buy_hold(df, initial_capital= 10000):
  """
  Calculate the portfolio value of a buy-and-hold investment.

  Parameters
  ---------
  df : pandas.Dataframe
      DataFrame containing daily stock returns.
    initial_capital : float
      Starting portfolio value.

  Returns
  ---------
  pandas.DataFrame
      DataFrame with BuyHold porfolio values.
  """
  df["BuyHold"] = (
      initial_capital
      * (1 + df["Returns"].fillna(0)).cumprod()
  )

  return df


In [ ]:
apple = calculate_buy_hold(apple)

In [ ]:
apple[["Returns", "BuyHold"]].tail()

In [ ]:
print("Strategy Final Value:",
      apple["Portfolio"].iloc[-1])
print("Buy-and-Hold Final Value:",
      apple["BuyHold"].iloc[-1])

# Step 6: Performance Metrics

We evaluate the strategy using total return, annualised volatility, Sharpe Ratio and maximum drawdown

In [ ]:
from functools import total_ordering
def calculate_metrics(df, return_column, portfolio_column, initial_capital=10000):
  """
  Calculate performance metrics of the strategy for a certain portfolio

  Parameters
  ---------
  df : pandas.DataFrame
      DataFrame containing portfolio and return data.
    return_column : str
      Column containing daily returns.
    portfolio_columns : str
      Column containing portfolio values.
    initial_capital : float
      Starting portfolio value.

  Returns
  ---------
  dict
      Dictionary containing perfomance metrics.
  """
  total_return = (
      df[portfolio_column].iloc[-1] - initial_capital
  ) / initial_capital

  # Volaititlty calculation
  daily_volatitlity = df[return_column].std() # Volaititlty calculation
  anualised_volatility = daily_volatitlity * np.sqrt(252)

  # Sharpe's Ratio Calculations
  mean_return = df[return_column].mean()

  sharpe_ratio = (
      mean_return / daily_volatitlity
  ) * np.sqrt(252) # assuming risk free rate

  # Maximum Drawdown
  running_peak = df[portfolio_column].cummax()

  drawdown = (
      df[portfolio_column] - running_peak
  ) / running_peak

  max_drawdown = drawdown.min()

  return {
      "Total Return": total_return,
      "Annualised Volatility": anualised_volatility,
      "Sharpe Ratio": sharpe_ratio,
      "Maximum Drawdown": max_drawdown
  }


In [ ]:
strategy_metrics = calculate_metrics(
    apple,
    return_column="Strategy_Returns",
    portfolio_column="Portfolio"
)

In [ ]:
strategy_metrics

In [ ]:
buyhold_metrics = calculate_metrics(
    apple,
    return_column="Returns",
    portfolio_column="BuyHold"
)


In [ ]:
buyhold_metrics


In [ ]:
results = pd.DataFrame({
    "Metric": strategy_metrics.keys(),
    "Strategy": strategy_metrics.values(),
    "Buy & Hold": buyhold_metrics.values()
})

results

In [ ]:
results_formatted = results.copy()

percentage_metrics = [
    "Total Return",
    "Annualised Volatility",
    "Maximum Drawdown"
]

for metric in percentage_metrics:
    results_formatted.loc[
        results_formatted["Metric"] == metric,
        ["Strategy", "Buy & Hold"]
    ] *= 100

results_formatted["Strategy"] = results_formatted["Strategy"].round(2)
results_formatted["Buy & Hold"] = results_formatted["Buy & Hold"].round(2)

results_formatted

In [ ]:
def create_results_table(strategy_metrics, buyhold_metrics):
  """
  Geerte a formatted comparison table for the strategy
  and Buy & Hold benchmark.
  """
  results = pd.DataFrame({
      "Strategy": strategy_metrics,
      "Buy & Hold": buyhold_metrics
  })

  percentage_metrics = [
      "Total Return",
      "Annualised Volatility",
      "Maximum Drawdown"
  ]

  for metric in percentage_metrics:
    results.loc[metric] *= 100

  results = results.round(2)

  results.index = [
      "Total Return (%)",
      "Annualised Volatility (%)",
      "Sharpe Ratio",
      "Maximum Drawdown (%)"
  ]

  return results


# THE BACKTEST PIPELINE

In [ ]:
def backtest(df, initial_capital=10000):
  """
  Run the complete moving-average backtest

  Parameters
  ---------
  df : pandas.DataFrame
      Stock price data
  initial_capital : float
      Starting portfolio value.

  Returns
  ---------
  pandas.DataFrame
      DataFrame containing containing the comlete backtest.
  strategy_metrics : dict
      Perfomance metrics of the strategy.
  buthold_metrics : dict
      Perfomance metrics of the buy-and-hold strategy.
  """
  # Add moving averages
  df = add_moving_averages(df)

  # Generate trading positions
  df = generate_signals(df)

  # Calculate daily and strategy returns
  df = calculate_returns(df)

  # Calculate strategy portfolio
  df = calculate_portfolio(df, initial_capital=initial_capital)

  # Calculate Buy & Hold portfolio
  df = calculate_buy_hold(df, initial_capital=initial_capital)

  # Calculate strategy metrics
  strategy_metrics = calculate_metrics(
      df,
      return_column="Strategy_Returns",
      portfolio_column="Portfolio",
      initial_capital=initial_capital
  )

  # Calculate Buy & Hold metrics
  buyhold_metrics = calculate_metrics(
      df,
      return_column="Returns",
      portfolio_column="BuyHold",
      initial_capital=initial_capital
  )

  # Create comparison table
  buyhold_metrics = calculate_metrics(
      df,
      return_column="Returns",
      portfolio_column="BuyHold",
      initial_capital=initial_capital
  )

  # Create comparison table
  results_table = create_results_table(
      strategy_metrics,
      buyhold_metrics
  )

  return {
        "data": df,
        "strategy_metrics": strategy_metrics,
        "buyhold_metrics": buyhold_metrics,
        "results_table": results_table
    }

# NOTE ON RUNNING BACKTEST


To search the results after running backtest(), use the following
functions:

- results["data"] --- gets the complete DataFrame
- results["strategy_metrics"] --- gets the strategy metrics
- results["buyhold_metrics"] --- gets the Buy & Hold metrics
- results["results_table"] --- gets the comparison table

This is to be used for your own named backtesting e.g:

results = backtest(apple.copy())

Also note that you can individual run each function such as calculate_metrics() and create_results_table() on their own as well.

# COMPARE STRATEGY PERFORMANCE

The strtagy is compared wth a buy & hold benchmark using four perfomance metrics:

- Total Return
- Annualised Volatility
- Sharpe Ratio
- Maximum Drawdown

Percentage-basd metrics are dislayed as percentages, while the Sharpe Ratio remains a numerical ratio.


In [ ]:
results = backtest(apple.copy())

In [ ]:
results["results_table"]

# Price and Moving Average Visualisation

This plot shows the stock's closing price together with the 30-day and 50-day moving averages used by the trading strategy.

The purpose is to visually examine the relationship between the moving averages and the strategy's trading signals.

In [ ]:
def plot_price_and_moving_averages(df):
  """
  Plot closing price with the 30-day and 50-day moving averages.
  """
  plt.figure(figsize=(14,7))

  plt.plot(
      df.index,
      df["Close"],
      label="Apple Closing Price"
  )

  plt.plot(
      df.index,
      df["MA30"],
      label="MA30"
  )

  plt.plot(
      df.index,
      df["MA50"],
      label="MA50"
  )

  # identify entries and exits
  signal_change = df["Position"].diff()

  entries = signal_change == 1
  exits = signal_change == -1

  # plot entry and exit points
  plt.scatter(
      df.index[entries],
      df.loc[entries, "Close"],
      marker="^",
      color="black",
      label="Entry"
  )

  plt.scatter(
      df.index[exits],
      df.loc[exits, "Close"],
      marker="v",
      color="purple",
      label="Exit"
  )

  plt.title("AAPL Price and Moving Averages")
  plt.xlabel("Date")
  plt.ylabel("Price ($)")
  plt.legend()
  plt.grid(True)

  plt.show()

In [ ]:
plot_price_and_moving_averages(results["data"])

In [ ]:
results["data"]["Position"].diff().value_counts()
df = results["data"].copy()

df["Signal_Change"] = df["Position"].diff()

df["Entry"] = df["Signal_Change"] == 1
df["Exit"] = df["Signal_Change"] == -1

In [ ]:
df[["Position", "Signal_Change", "Entry", "Exit"]].tail(20)

# Portfolio Growth

This plot compares the growth of a $10,00 investment using the moving average strategy against buy-and-hold investment.

The purpose is to examine how the two approaches performed over time and identify periods where their perfomance diverged.

In [ ]:
def plot_portfolio_growth(df):
    """
    Plot strategy portfolio value against Buy & Hold.
    """

    plt.figure(figsize=(14, 7))

    plt.plot(
        df.index,
        df["Portfolio"],
        label="Moving Average Strategy"
    )

    plt.plot(
        df.index,
        df["BuyHold"],
        label="Buy & Hold"
    )

    plt.title("Portfolio Growth: Strategy vs Buy & Hold")
    plt.xlabel("Date")
    plt.ylabel("Portfolio Value ($)")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
plot_portfolio_growth(results["data"])

# Drawdown Analysis

Drawdown measures the percentage decline in portfolio value from the previous highest portfolio value.

This visualisation showds the periods in which the strategy and Buy & Hold experienced declines from their previous peaks.

In [ ]:
def plot_drawdown(df):
    """
    Plot portfolio drawdowns for the strategy and Buy & Hold.
    """

    strategy_peak = df["Portfolio"].cummax()
    buyhold_peak = df["BuyHold"].cummax()

    strategy_drawdown = (
        df["Portfolio"] - strategy_peak
    ) / strategy_peak

    buyhold_drawdown = (
        df["BuyHold"] - buyhold_peak
    ) / buyhold_peak

    plt.figure(figsize=(14, 7))

    plt.plot(
        df.index,
        strategy_drawdown,
        label="Moving Average Strategy"
    )

    plt.plot(
        df.index,
        buyhold_drawdown,
        label="Buy & Hold"
    )

    plt.title("Portfolio Drawdown")
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
plot_drawdown(results["data"])

In [ ]:
"""
LOADING THE MICROSOFT DATA
"""
msft = yf.download(
    "MSFT",
    start="2020-01-01",
    end="2025-01-01",
    auto_adjust=True
)

[*********************100%***********************]  1 of 1 completed


In [ ]:
msft.tail()

Price,Close,High,Low,Open,Volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT
Date,,,,,
2024-12-24,433.363525,433.629878,428.293346,428.747091,7164500
2024-12-26,432.160095,434.951678,430.700214,433.116923,8194200
2024-12-27,424.683014,429.309322,420.559789,428.697747,18117700
2024-12-30,419.060455,421.743517,416.170254,420.273762,13158700
2024-12-31,415.775696,420.934679,414.947107,420.313230,13246500


In [ ]:
msft.columns = msft.columns.get_level_values(0)

In [ ]:
msft.shape

(1258, 5)

In [53]:
msft.columns

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

In [54]:
msft_results = backtest(msft.copy())

In [55]:
msft_results['results_table']

,Strategy,Buy & Hold
Total Return (%),8.82,174.36
Annualised Volatility (%),21.07,30.50
Sharpe Ratio,0.19,0.82
Maximum Drawdown (%),-35.14,-37.15
